Built based on the paper: [A text-embedding-based approach to measuring patent-to-patent technological similarity](https://www.sciencedirect.com/science/article/pii/S0040162522000919?ref=pdf_download&fr=RR-2&rr=a5509ebaa8342bcd)

In [1]:
import gc
import itertools
import logging
import os
import tomllib
from pathlib import Path

import gensim
import h5py
import nltk
import numpy as np
import pandas as pd
import scipy
import tqdm
from annoy import AnnoyIndex
from gensim.corpora import Dictionary
from gensim.matutils import sparse2full
from gensim.models import Word2Vec

# from gensim.test.utils import datapath
from gensim.models.phrases import ENGLISH_CONNECTOR_WORDS, Phrases
from gensim.models.tfidfmodel import TfidfModel
from nltk import sent_tokenize

In [2]:
nltk.download('punkt_tab')

logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\MXP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
with open("./config.toml", "rb") as f:
    config = tomllib.load(f)

INPUT_FILE = Path(config["path"]["input"]) / "CN_abstracts.parquet"
OUTPUT = Path(config["path"]["output"])
OUTPUT.mkdir(parents=True, exist_ok=True)
INTERMED = Path(config["path"]["intermed"])
INTERMED.mkdir(parents=True, exist_ok=True)

# hyps
bi_min_count = config["bigram"]["min_count"]
bi_threshold = config["bigram"]["threshold"]
vector_size = config["word2vec"]["vector_size"]
window = config["word2vec"]["window"]
w2v_min_count = config["word2vec"]["min_count"]
epochs = config["word2vec"]["epochs"]
trees = config["similarity"]["trees"]
search_k = config["similarity"]["search_k"]
thr = config["similarity"]["thr"]
upto = config["similarity"]["upto"]

WORKERS = int(max(os.cpu_count()/2, 1))

In [4]:
print('Nr of workers used = ', WORKERS)

Nr of workers used =  6


In [5]:
# Preprocessing (sentence tokenizer + basik Gensim - lowercases, tokenizes, de-accents (optional))
def prepro(x):
    x = sent_tokenize(x)
    clean = []
    for j in x:
        clean.append(gensim.utils.simple_preprocess(j, min_len=1)) # min_len=1 -> keeps single letter token
    return clean

def bigrammer(x):
    sents = []
    for i in x:
        sents.append(bigram[i])
    return sents

def tfidfer(para):
    return model_tfidf[docs_dict.doc2bow(itertools.chain(*para))]

def preprocess_1 (data):
    data['appln_abstract_prepro'] = list(map(prepro, data['abstract']))
    return data

def preprocess_2 (data):
    data['appln_abstract_prepro_bi'] = list(map(bigrammer, data['appln_abstract_prepro']))
    return data

def preprocess_3 (data):
    data['tfidf'] = list(map(tfidfer, data['appln_abstract_prepro_bi']))
    return data

def ang2cos(d):
    # Annoy 'angular' distance d = sqrt(2 - 2*cos)  ->  cosine similarity (paper Eq. 1)
    return 1 - np.asarray(d) ** 2 / 2

In [6]:
data = pd.read_csv('https://raw.githubusercontent.com/AI-Growth-Lab/Patent_p2p_similarity_w2v/main/patent_dtatset_sample_1k.csv')

In [7]:
data = data[["appln_id", "appln_abstract"]]
data = data.rename(columns={"appln_abstract": "abstract"})

In [8]:
print('Total appln to process = ', len(data))

Total appln to process =  1000


In [9]:
data = preprocess_1(data) # add prepro

In [10]:
# build bigrams
sentences = [sent for doc in data['appln_abstract_prepro'] for sent in doc if sent]
bigram = Phrases(sentences, min_count=bi_min_count, threshold=bi_threshold, connector_words=ENGLISH_CONNECTOR_WORDS)
bigram = gensim.models.phrases.Phraser(bigram)
bigram.save(str(INTERMED / 'bigram.m'))

2026-11-04 05:06:24,624 : INFO : collecting all words and their counts
2026-11-04 05:06:24,626 : INFO : PROGRESS: at sentence #0, processed 0 words and 0 word types
2026-11-04 05:06:24,868 : INFO : collected 90708 token types (unigram + bigrams) from a corpus of 179506 words and 7674 sentences
2026-11-04 05:06:24,869 : INFO : merged Phrases<90708 vocab, min_count=2, threshold=1, max_vocab_size=40000000>
2026-11-04 05:06:24,870 : INFO : Phrases lifecycle event {'msg': 'built Phrases<90708 vocab, min_count=2, threshold=1, max_vocab_size=40000000> in 0.25s', 'datetime': '2026-11-04T05:06:24.870104', 'gensim': '4.4.0', 'python': '3.12.16 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 20:13:05) [MSC v.1942 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26300-SP0', 'event': 'created'}
2026-11-04 05:06:24,871 : INFO : exporting phrases from Phrases<90708 vocab, min_count=2, threshold=1, max_vocab_size=40000000>
2026-11-04 05:06:25,098 : INFO : FrozenPhrases lifecycle event {'msg': 'export

In [11]:
data = preprocess_2(data) # add bigram

In [12]:
# word to vec
sentences = [sent for doc in data['appln_abstract_prepro_bi'] for sent in doc if sent] # using bigram
model_with_phrases = Word2Vec(sentences=sentences, vector_size=vector_size, window=window,
                              min_count=w2v_min_count, epochs=epochs, workers=WORKERS)
model_with_phrases.save(str(INTERMED / 'w2v.m'))

2026-11-04 05:06:25,358 : INFO : collecting all words and their counts
2026-11-04 05:06:25,359 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-11-04 05:06:25,410 : INFO : collected 14366 word types from a corpus of 155332 raw words and 7674 sentences
2026-11-04 05:06:25,412 : INFO : Creating a fresh vocabulary
2026-11-04 05:06:25,505 : INFO : Word2Vec lifecycle event {'msg': 'effective_min_count=1 retains 14366 unique words (100.00% of original 14366, drops 0)', 'datetime': '2026-11-04T05:06:25.505437', 'gensim': '4.4.0', 'python': '3.12.16 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 20:13:05) [MSC v.1942 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26300-SP0', 'event': 'prepare_vocab'}
2026-11-04 05:06:25,506 : INFO : Word2Vec lifecycle event {'msg': 'effective_min_count=1 leaves 155332 word corpus (100.00% of original 155332, drops 0)', 'datetime': '2026-11-04T05:06:25.506407', 'gensim': '4.4.0', 'python': '3.12.16 | packaged by Anaconda, Inc.

In [13]:
# Build Gensim Dictionary
voc = model_with_phrases.wv.index_to_key
docs_dict = Dictionary([voc])
docs_dict.compactify()
docs_dict.save(str(INTERMED / 'docs_dict_11_2.d'))

2026-11-04 05:06:26,609 : INFO : adding document #0 to Dictionary<0 unique tokens: []>
2026-11-04 05:06:26,692 : INFO : built Dictionary<14366 unique tokens: ['a', 'abel', 'aberration', 'ability', 'able']...> from 1 documents (total 14366 corpus positions)
2026-11-04 05:06:26,694 : INFO : Dictionary lifecycle event {'msg': "built Dictionary<14366 unique tokens: ['a', 'abel', 'aberration', 'ability', 'able']...> from 1 documents (total 14366 corpus positions)", 'datetime': '2026-11-04T05:06:26.694840', 'gensim': '4.4.0', 'python': '3.12.16 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 20:13:05) [MSC v.1942 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26300-SP0', 'event': 'created'}
2026-11-04 05:06:26,710 : INFO : Dictionary lifecycle event {'fname_or_handle': 'intermed\\docs_dict_11_2.d', 'separately': 'None', 'sep_limit': 10485760, 'ignore': frozenset(), 'datetime': '2026-11-04T05:06:26.710602', 'gensim': '4.4.0', 'python': '3.12.16 | packaged by Anaconda, Inc. | (main, Oct 26 

In [14]:
# tfidf at patent level
docs = [list(itertools.chain(*doc)) for doc in data['appln_abstract_prepro_bi']]
model_tfidf = TfidfModel((docs_dict.doc2bow(d) for d in docs), id2word=docs_dict)
model_tfidf.save(str(INTERMED / 'model_tfidf_11_2.m'))

docs_vecs = (sparse2full(model_tfidf[docs_dict.doc2bow(d)], len(docs_dict)) for d in docs) # generator

2026-11-04 05:06:26,754 : INFO : collecting document frequencies
2026-11-04 05:06:26,756 : INFO : PROGRESS: processing document #0
2026-11-04 05:06:26,920 : INFO : TfidfModel lifecycle event {'msg': 'calculated IDF weights for 1000 documents and 14366 features (85836 matrix non-zeros)', 'datetime': '2026-11-04T05:06:26.920360', 'gensim': '4.4.0', 'python': '3.12.16 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 20:13:05) [MSC v.1942 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26300-SP0', 'event': 'initialize'}
2026-11-04 05:06:26,921 : INFO : TfidfModel lifecycle event {'fname_or_handle': 'intermed\\model_tfidf_11_2.m', 'separately': 'None', 'sep_limit': 10485760, 'ignore': frozenset(), 'datetime': '2026-11-04T05:06:26.921872', 'gensim': '4.4.0', 'python': '3.12.16 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 20:13:05) [MSC v.1942 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26300-SP0', 'event': 'saving'}
2026-11-04 05:06:26,992 : INFO : saved intermed\model_tfidf_11_2.m

In [15]:
data = preprocess_3(data)
data.to_parquet(INTERMED / "data.parquet", engine="pyarrow", index=False)

In [16]:
# Selected Word-Embeddings
emb_vecs_selftrained = np.vstack([model_with_phrases.wv[docs_dict.get(i)] for i in range(len(docs_dict))]) # n*300
n_abstracts = len(data)
appln_ids = data['appln_id'].to_numpy()          # same order as docs / docs_vecs

with h5py.File(INTERMED / 'docvecs_23_4_test.h5', 'w') as h5f:     # 'w' = overwrite, safe to rerun
    dataset = h5f.create_dataset('weighted_tfidf_docvecs', (n_abstracts, vector_size), dtype='f4')
    h5f.create_dataset('appln_id', data=appln_ids)

    a = np.vstack([next(docs_vecs) for _ in range(n_abstracts)])
    dataset[:] = np.dot(a, emb_vecs_selftrained)

In [17]:
h5f = h5py.File(INTERMED / 'docvecs_23_4_test.h5', 'r')
vecs = h5f['weighted_tfidf_docvecs'] # dim 100*300
ids = h5f['appln_id'][:] # get every element
row_of = {aid: i for i, aid in enumerate(ids)}      # appln_id -> row

def vec(appln_id):
    return vecs[row_of[appln_id]]

scipy.spatial.distance.cosine(vec(ids[9]), vec(ids[10]))   # or vec(<a real appln_id>)

np.float32(1.0669231e-05)

In [18]:
# ---similarity search

In [19]:
# build trees and save the file
t = AnnoyIndex(vector_size, 'angular')
t.on_disk_build(str(INTERMED / 'vecs_build.annoy'))
for row in tqdm.tqdm(range(len(vecs))):
    t.add_item(row, vecs[row])       # Annoy item = row number
t.build(trees)
t.unload()                           # release the file (needed on Windows before re-loading)

100%|██████████| 1000/1000 [00:00<00:00, 15401.49it/s]


True

In [20]:
tt = AnnoyIndex(vector_size, 'angular')
tt.load(str(INTERMED / 'vecs_build.annoy'), prefault=False)

simH5 = h5py.File(str(OUTPUT / 'similarities_23_2.h5'), 'w')
final = simH5.create_dataset('sims_50',(10000000,3), maxshape=(None,3), dtype='f8', compression="gzip")

In [21]:
pbar = tqdm.tqdm(total=tt.get_n_items())

blocks = []
start = 0
end = 0
counter = 0
for i in range(tt.get_n_items()):
    j = 100
    tresh = 1.0
    while tresh >= thr:
        y = tt.get_nns_by_item(i, j, search_k=search_k, include_distances=True)
        sims = ang2cos(y[1])
        tresh = sims[-1]
        if len(y[0]) < j:
            break
        if j >= upto:
            break
        j += 50
    keep = sims >= thr                     # paper: discard pairs below 0.65
    block = np.column_stack((np.full(keep.sum(), i), np.asarray(y[0])[keep], sims[keep]))
    end = end + len(block)
    blocks.append(block)
    counter += 1
    if counter == 1000:
        counter = 0
        blocks_np = np.vstack(blocks)
        if end >= final.shape[0]:
            final.resize((end+(end - final.shape[0]),3))
        final[start:end] = blocks_np
        blocks = []
        gc.collect()
        start = end
        pbar.update(1000)

if blocks:
    blocks_np = np.vstack(blocks)
    if end > final.shape[0]:
        final.resize((end+(end - final.shape[0]),3))
    final[start:end] = blocks_np
    pbar.update(counter)
    blocks = []

# trim the unused pre-allocated rows of zeros
final.resize((end, 3))

pbar.close()
simH5.close()

100%|██████████| 1000/1000 [00:01<00:00, 932.75it/s]


In [22]:
def lookup(x, j):
    # index, similar patents
    y = tt.get_nns_by_item(x, j, search_k=search_k, include_distances=True)
    sims = ang2cos(y[1])
    return np.array(list(zip([x for _ in range(len(y[0]))],y[0],sims)))

In [23]:
lookup(0, 100)

array([[  0.        ,   0.        ,   0.99999999],
       [  0.        , 706.        ,   0.99997757],
       [  0.        ,  96.        ,   0.99997604],
       [  0.        , 629.        ,   0.99997573],
       [  0.        , 413.        ,   0.99997387],
       [  0.        , 589.        ,   0.99997378],
       [  0.        , 613.        ,   0.99997329],
       [  0.        , 943.        ,   0.99997111],
       [  0.        , 313.        ,   0.99997051],
       [  0.        , 116.        ,   0.99997045],
       [  0.        , 102.        ,   0.99996982],
       [  0.        , 651.        ,   0.99996947],
       [  0.        , 316.        ,   0.99996863],
       [  0.        , 760.        ,   0.99996859],
       [  0.        , 653.        ,   0.99996855],
       [  0.        ,  11.        ,   0.99996748],
       [  0.        , 107.        ,   0.999967  ],
       [  0.        , 998.        ,   0.99996665],
       [  0.        ,  42.        ,   0.99996602],
       [  0.        , 654.     